In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import numpy as np

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

data = pd.read_csv('train.csv')

X = data.drop('Survived', axis=1)
y = data['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=RANDOM_STATE, stratify=y
)

baseline_pred = np.zeros_like(y_test)
baseline_f1 = f1_score(y_test, baseline_pred, zero_division=0)
print(f'Baseline F1-score: {baseline_f1:.4f}')

numeric_features = ['Age', 'Fare']
numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])

categorical_features = ['Sex', 'Pclass', 'Embarked']
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot',OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features),('cat', categorical_transformer, categorical_features)])

model = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', LogisticRegression(random_state=RANDOM_STATE))])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
model_f1 = f1_score(y_test, y_pred)
print(f'Model F1-score: {model_f1:.4f}')

if hasattr(model.named_steps['classifier'], 'coef_'):
  feature_names = (model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features))
  all_features = numeric_features + list(feature_names)
  coef = model.named_steps['classifier'].coef_[0]
  print('n\Feature importance: ')
  for feature, importance in zip(all_features, coef):
    print(f'{feature}:{importance:.4f}')

Baseline F1-score: 0.0000
Model F1-score: 0.6761
n\Feature importance: 
Age:-0.0364
Fare:0.0009
Sex_female:1.5377
Sex_male:-0.9834
Pclass_1:1.2169
Pclass_2:0.3029
Pclass_3:-0.9655
Embarked_C:0.3307
Embarked_Q:0.3515
Embarked_S:-0.1279


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Выбрал метрику F1-score так как данные Survived несбалансированы и важно точность предсказания по выжившим.
Модель LpgisticRegression выбрана как простая интерпретируемая модель бинарной классификации
